Load the required Libraries

In [37]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

Load the HuggingFace Dataset

In [38]:
dataset = load_dataset("drorrabin/phishing_emails-data")

dataset

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})

Inspect the Dataset

In [39]:
print(dataset)
print(dataset["train"].column_names)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'email_type'],
        num_rows: 26946
    })
    test: Dataset({
        features: ['text', 'email_type'],
        num_rows: 3705
    })
})
['text', 'email_type']
{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.c

Text column name is 'text'
Label column name is 'email type'

In [40]:
dataset["train"].to_pandas()["email_type"].value_counts()

email_type
phishing email    13473
safe email        13473
Name: count, dtype: int64

Label Distribution

In [41]:
def encode_labels(example):
    if example["email_type"] == "phishing email":
        example["labels"] = 1
    else:
        example["labels"] = 0
    return example

dataset = dataset.map(encode_labels)

In [42]:
dataset = dataset.remove_columns(["email_type"])

Load BERT Tokenizer

In [43]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

Tokenize Dataset

In [44]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/26946 [00:00<?, ? examples/s]

Map:   0%|          | 0/3705 [00:00<?, ? examples/s]

Remove Unnecessary Columns and make the dataset Py-torch ready

In [45]:

tokenized_dataset.set_format("torch")

Now load the BERT model

In [46]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


HuggingFace:

Loads pretrained BERT encoder

Adds a new classification head (classifier.weight, classifier.bias)

Randomly initializes that head

In [47]:
tokenized_dataset["train"][0]

{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08\n\nEmail type is: phishing email',
 'labels': tensor(1),
 'input_ids': tensor([  101,  2003,  1996,  2206, 10373,  3647,  2030, 13569, 12227,  1029,
          1029,  3

In [49]:
set(dataset["train"]["labels"])

{0, 1}

In [50]:
tokenized_dataset.set_format("torch")

In [51]:
tokenized_dataset["train"][0]

{'text': 'Is the following email safe or phishing??\n\nDate: Thu, 07 Aug 2008 10:59:35 +0200\n\nSender: Daily Top 10 <avadivap_1981@techsult.com>\n\nReceiver: user2.12@gvc.ceas-challenge.cc\n\nEmail Subject: CNN.com Daily Top 10\n\nEmail Body: THE DAILY TOP 10 from CNN.com Top videos and stories as of: Aug  1, 2008  3:58 PM EDT  TOP 10 VIDEOS 1. PARIS HILTON TAKES ON MCCAIN http://www.cnn.com/video/partners/email/index.html?url/video/politics/2008/08/06/wynter.paris.hilton.ad.cnn Paris Hilton swings back at Republican presidential candidate John McCain. Kareen Wynter reports. 2. BIKINI BARISTA STAND CLOSED http://www.cnn.com/video/partners/email/index.html?url/video/living/2008/08/06/pkg.bikini.baristas.barred.kiro 3. TOT GRANDMA REACTS TO CHARGES http://www.cnn.com/video/partners/email/index.html?url/video/crime/2008/08\n\nEmail type is: phishing email',
 'labels': tensor(1),
 'input_ids': tensor([  101,  2003,  1996,  2206, 10373,  3647,  2030, 13569, 12227,  1029,
          1029,  3

Define Metrics (Critical for Cybersecurity)

In [52]:
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    return {
        "accuracy": metric_accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": metric_f1.compute(predictions=predictions, references=labels)["f1"],
        "precision": metric_precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": metric_recall.compute(predictions=predictions, references=labels)["recall"],
    }

In phishing detection, recall is critical (false negatives are dangerous).

Define Training Arguments

In [53]:
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="tensorboard"   # ✅ new way
)

In [54]:
torch.cuda.is_available()

False

Initialize Trainer

In [55]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)